In [0]:
from pyspark.sql.functions import *

df = spark.table("gizmo_box.bronze.v_orders")

df_new = df.select(get_json_object(col("value"), "$.order_id").alias("order_id"), get_json_object(col("value"), "$.items").alias("items"))


display(df_new)

In [0]:
%sql 

-- select value:order_id, value:items[0]:details.brand as brand,value from gizmo_box.bronze.v_orders;

select value:order_id, value:items[0]:price::Integer as price,value from gizmo_box.bronze.v_orders;

## Fixed the data using regex

In [0]:
%sql
create or replace temp view v_orders_fix as
select regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date" : "\$1"') as value from gizmo_box.bronze.v_orders;

In [0]:
%sql

CREATE TABLE gizmo_box.silver.orders_json
AS
select from_json(value, "STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>") as value from v_orders_fix

In [0]:
%sql

create or replace temp view order_explode as
select value.customer_id,explode(array_distinct(value.items)) item, value.order_date, value.order_id, value.order_status, value.payment_method, value.total_amount, value.transaction_timestamp from gizmo_box.silver.orders_json;




In [0]:
%sql

CREATE TABLE gizmo_box.silver.orders 
as
select customer_id, item.item_id,item.name, item.price, item.quantity, item.details.brand, item.details.color, order_date, order_id, order_status, payment_method, total_amount, transaction_timestamp from order_explode

In [0]:
%sql
Select * from gizmo_box.silver.orders 